# Lista 03 — Inteligência Artificial
## Titanic: imputação, balanceamento e classificação

Esta solução segue todas as etapas do enunciado e as orientações do quadro:

1. abre e inspeciona o arquivo;
2. codifica os atributos;
3. separa 80% para treino e 20% para teste;
4. ajusta imputação e padronização **somente no treino**;
5. aplica SMOTE ou RandomUnderSampling **somente no treino**;
6. compara KNN Imputer e MissForest, Árvore de Decisão e Random Forest, Random Search e Optuna.

**Cuidados metodológicos:** `boat` e `body` foram removidos porque são informações conhecidas depois do desastre e vazariam a resposta. `cabin` e `home.dest` foram retirados por terem muitas ausências; `name` e `ticket` são identificadores de alta cardinalidade. A classe positiva é `survived = 1`.

Se necessário, a próxima célula instala automaticamente `imbalanced-learn` e `optuna`.

In [ ]:
import importlib.util
import subprocess
import sys

dependencias = {
    'imblearn': 'imbalanced-learn',
    'optuna': 'optuna',
}
faltantes = [pacote for modulo, pacote in dependencias.items() if importlib.util.find_spec(modulo) is None]
if faltantes:
    print('Instalando dependências:', ', '.join(faltantes))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *faltantes])
    print('Instalação concluída.')
else:
    print('Dependências já instaladas.')

In [ ]:
from pathlib import Path
import sys
import time
import warnings
import json

BASE = Path.cwd()
if not (BASE / 'titanic completo.csv').exists() and (BASE / 'Lista03').exists():
    BASE = BASE / 'Lista03'
DEPS = BASE / '.deps'
if DEPS.exists():
    sys.path.insert(0, str(DEPS))

import numpy as np
import pandas as pd
import matplotlib
try:
    get_ipython
except NameError:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import KNNImputer, IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.base import clone
from scipy.stats import ks_2samp, randint, uniform

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
import optuna

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
RESULTADOS = BASE / 'resultados_lista03'
RESULTADOS.mkdir(exist_ok=True)

def mostrar(obj):
    try:
        display(obj)
    except NameError:
        print(obj.to_string() if hasattr(obj, 'to_string') else obj)

## 1. Leitura, inspeção e codificação

In [ ]:
arquivo = BASE / 'titanic completo.csv'
base_original = pd.read_csv(arquivo)
print('Dimensões:', base_original.shape)
mostrar(base_original.head())

ausencias = base_original.isna().sum().to_frame('ausentes')
ausencias['percentual'] = 100 * ausencias['ausentes'] / len(base_original)
mostrar(ausencias.query('ausentes > 0').sort_values('percentual', ascending=False).round(2))

atributos = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
X = base_original[atributos].copy()
y = base_original['survived'].astype(int)

# Codificação explícita e reproduzível antes da separação.
X['sex'] = X['sex'].map({'male': 0, 'female': 1})
X['embarked'] = X['embarked'].map({'C': 0, 'Q': 1, 'S': 2})

print('\nAtributos codificados:')
mostrar(X.head())
distribuicao_classe = y.value_counts().sort_index().rename(index={0: 'Morreu', 1: 'Sobreviveu'}).to_frame('quantidade')
distribuicao_classe['percentual'] = 100 * distribuicao_classe['quantidade'] / len(y)
mostrar(distribuicao_classe.round(2))

fig, ax = plt.subplots(figsize=(7, 5))
cores = ['#3b7ca6', '#e58b3c']
contagens = y.value_counts().sort_index()
barras = ax.bar(['Não sobreviveu (0)', 'Sobreviveu (1)'], contagens.values, color=cores)
ax.bar_label(barras, labels=[f'{v} ({v / len(y):.1%})' for v in contagens.values], padding=4)
ax.set_title('Distribuição da variável sobrevivência')
ax.set_xlabel('Classe')
ax.set_ylabel('Quantidade')
ax.set_ylim(0, contagens.max() * 1.15)
plt.tight_layout()
plt.savefig(RESULTADOS / '01_distribuicao_classe.png', dpi=160, bbox_inches='tight')
plt.show()

## 2. Separação treino/teste

São usados 80% para treino e 20% para teste, com estratificação. A separação vem antes do ajuste dos imputadores, do `StandardScaler` e dos balanceadores. Assim, nenhuma informação do teste participa do aprendizado.

In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print('Treino:', X_treino.shape, '| Teste:', X_teste.shape)
print('Proporção de sobreviventes — treino:', f'{y_treino.mean():.2%}', '| teste:', f'{y_teste.mean():.2%}')

## 3. Imputação: KNN versus MissForest

`KNNImputer` estima os ausentes pelos vizinhos mais próximos. O segundo método implementa a ideia do **MissForest** solicitada no enunciado: `IterativeImputer` com um conjunto de árvores (`ExtraTreesRegressor`). Cada método é ajustado apenas com `X_treino` e depois aplicado ao teste.

A tabela compara a distribuição observada do treino com a distribuição após a imputação. A estatística KS menor indica maior semelhança entre as distribuições. Para atributos categóricos codificados (`sex` e `embarked`), valores eventualmente contínuos gerados pelo imputador são arredondados e limitados às categorias válidas.

In [ ]:
def novo_imputador(nome):
    if nome == 'KNN':
        return KNNImputer(n_neighbors=5, weights='distance')
    return IterativeImputer(
        estimator=ExtraTreesRegressor(n_estimators=40, random_state=RANDOM_STATE, n_jobs=1),
        max_iter=8, initial_strategy='median', random_state=RANDOM_STATE, skip_complete=True
    )

def corrigir_categorias(df):
    df = df.copy()
    df['sex'] = df['sex'].round().clip(0, 1)
    df['embarked'] = df['embarked'].round().clip(0, 2)
    df['pclass'] = df['pclass'].round().clip(1, 3)
    return df

imputados = {}
linhas_imputacao = []
for nome in ['KNN', 'MissForest']:
    imp = novo_imputador(nome)
    treino_imp = pd.DataFrame(imp.fit_transform(X_treino), columns=X.columns, index=X_treino.index)
    teste_imp = pd.DataFrame(imp.transform(X_teste), columns=X.columns, index=X_teste.index)
    treino_imp = corrigir_categorias(treino_imp)
    teste_imp = corrigir_categorias(teste_imp)
    imputados[nome] = (imp, treino_imp, teste_imp)
    for atributo in ['age', 'fare', 'embarked']:
        observado = X_treino[atributo].dropna()
        depois = treino_imp[atributo]
        linhas_imputacao.append({
            'imputador': nome, 'atributo': atributo,
            'ausentes_antes': int(X_treino[atributo].isna().sum()),
            'media_antes': observado.mean(), 'media_depois': depois.mean(),
            'mediana_antes': observado.median(), 'mediana_depois': depois.median(),
            'desvio_antes': observado.std(), 'desvio_depois': depois.std(),
            'KS': ks_2samp(observado, depois).statistic,
        })

tabela_imputacao = pd.DataFrame(linhas_imputacao)
tabela_imputacao.to_csv(RESULTADOS / 'comparacao_imputacao.csv', index=False, encoding='utf-8-sig')
mostrar(tabela_imputacao.round(3))

fig, eixos = plt.subplots(1, 3, figsize=(16, 4))
for ax, atributo in zip(eixos, ['age', 'fare', 'embarked']):
    sns.histplot(X_treino[atributo].dropna(), stat='density', element='step', fill=False, label='Observado', ax=ax)
    for nome in ['KNN', 'MissForest']:
        sns.kdeplot(imputados[nome][1][atributo], label=nome, ax=ax, warn_singular=False)
    ax.set_title(f'Distribuição de {atributo}')
    ax.legend()
plt.tight_layout()
plt.savefig(RESULTADOS / 'distribuicoes_imputacao.png', dpi=160, bbox_inches='tight')
plt.savefig(RESULTADOS / '02_distribuicoes_imputacao.png', dpi=160, bbox_inches='tight')
plt.show()

**Critério de escolha:** para a discussão final, é adotado o imputador com menor média da estatística KS entre os atributos avaliados, pois foi o que menos alterou a distribuição observada. Mesmo assim, ambos seguem para a grade experimental exigida no item 4.a.

In [ ]:
ks_medio = tabela_imputacao.groupby('imputador')['KS'].mean().sort_values()
imputador_adotado = ks_medio.index[0]
print('KS médio por método:')
mostrar(ks_medio.to_frame())
print('\nImputador adotado pelo critério de preservação:', imputador_adotado)

## 4. Balanceamento: SMOTE versus RandomUnderSampling

A classe sobrevivente é minoritária. O SMOTE cria exemplos sintéticos dessa classe; o `RandomUnderSampler` reduz aleatoriamente a classe majoritária. O balanceamento é aplicado somente ao treino já imputado e padronizado. A tabela abaixo é apenas descritiva; durante a otimização, as mesmas operações ficam dentro do pipeline e de cada dobra da validação cruzada.

In [ ]:
balanceadores = {
    'SMOTE': SMOTE(random_state=RANDOM_STATE, k_neighbors=5),
    'RandomUnderSampling': RandomUnderSampler(random_state=RANDOM_STATE),
}
linhas_balanceamento = []
for nome_imp, (_, treino_imp, _) in imputados.items():
    scaler = StandardScaler()
    treino_pad = scaler.fit_transform(treino_imp)  # fit somente no treino
    antes = y_treino.value_counts().sort_index()
    for nome_bal, bal in balanceadores.items():
        _, y_res = clone(bal).fit_resample(treino_pad, y_treino)
        depois = pd.Series(y_res).value_counts().sort_index()
        linhas_balanceamento.append({
            'imputador': nome_imp, 'balanceamento': nome_bal,
            'antes_morreu': int(antes.get(0, 0)), 'antes_sobreviveu': int(antes.get(1, 0)),
            'depois_morreu': int(depois.get(0, 0)), 'depois_sobreviveu': int(depois.get(1, 0)),
        })
tabela_balanceamento = pd.DataFrame(linhas_balanceamento)
tabela_balanceamento.to_csv(RESULTADOS / 'comparacao_balanceamento.csv', index=False, encoding='utf-8-sig')
mostrar(tabela_balanceamento)

fig, eixos = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
cenarios = {
    'Antes do balanceamento': [647, 400],
    'Depois do SMOTE': [647, 647],
    'Depois do RandomUnderSampling': [400, 400],
}
for ax, (titulo, valores) in zip(eixos, cenarios.items()):
    barras = ax.bar(['Não sobreviveu', 'Sobreviveu'], valores, color=['#3b7ca6', '#e58b3c'])
    ax.bar_label(barras, padding=3)
    ax.set_title(titulo)
    ax.set_xlabel('Classe')
    ax.tick_params(axis='x', rotation=12)
eixos[0].set_ylabel('Quantidade no treino')
plt.tight_layout()
plt.savefig(RESULTADOS / '03_balanceamento_classes.png', dpi=160, bbox_inches='tight')
plt.show()

## 5. Otimização e modelagem

Serão avaliadas as oito combinações: 2 imputadores × 2 balanceadores × 2 modelos. Em cada combinação são usados dois otimizadores:

- `RandomizedSearchCV` (Random Search);
- Optuna com amostrador TPE (otimização Bayesiana).

A métrica de seleção é F1 da classe sobrevivente, adequada ao desbalanceamento. Ambos usam exatamente as mesmas 3 dobras estratificadas e 15 tentativas. O pipeline garante que imputação, padronização e balanceamento sejam aprendidos apenas na parcela de treino de cada dobra.

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
N_TENTATIVAS = 15

def novo_modelo(nome):
    if nome == 'Árvore de Decisão':
        return DecisionTreeClassifier(random_state=RANDOM_STATE)
    return RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1)

def pipeline_experimento(nome_imp, nome_bal, nome_modelo):
    return Pipeline([
        ('imputer', novo_imputador(nome_imp)),
        ('scaler', StandardScaler()),
        ('sampler', clone(balanceadores[nome_bal])),
        ('model', novo_modelo(nome_modelo)),
    ], memory=str(RESULTADOS / '.pipeline_cache'))

def espaco_random(nome_modelo):
    comum = {
        'model__max_depth': [None, 3, 4, 5, 6, 8, 10, 12],
        'model__min_samples_split': randint(2, 21),
        'model__min_samples_leaf': randint(1, 11),
        'model__criterion': ['gini', 'entropy', 'log_loss'],
    }
    if nome_modelo == 'Random Forest':
        comum.update({
            'model__n_estimators': randint(80, 251),
            'model__max_features': ['sqrt', 'log2', None, 0.7],
        })
    return comum

def parametros_optuna(trial, nome_modelo):
    params = {
        'model__max_depth': trial.suggest_categorical('max_depth', [None, 3, 4, 5, 6, 8, 10, 12]),
        'model__min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'model__min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'model__criterion': trial.suggest_categorical('criterion', ['gini', 'entropy', 'log_loss']),
    }
    if nome_modelo == 'Random Forest':
        params.update({
            'model__n_estimators': trial.suggest_int('n_estimators', 80, 250),
            'model__max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None, 0.7]),
        })
    return params

def limpar_params(params):
    return {chave.replace('model__', ''): valor for chave, valor in params.items()}

def metricas_teste(modelo, X_avaliacao, y_real):
    pred = modelo.predict(X_avaliacao)
    return {
        'acuracia': accuracy_score(y_real, pred),
        'precisao': precision_score(y_real, pred, zero_division=0),
        'recall': recall_score(y_real, pred, zero_division=0),
        'f1': f1_score(y_real, pred, zero_division=0),
        'matriz_confusao': confusion_matrix(y_real, pred).tolist(),
    }

In [ ]:
resultados_metricas = []
resultados_buscas = []
modelos_ajustados = {}

for nome_imp in ['KNN', 'MissForest']:
    for nome_bal in ['SMOTE', 'RandomUnderSampling']:
        for nome_modelo in ['Árvore de Decisão', 'Random Forest']:
            chave_base = (nome_imp, nome_bal, nome_modelo)
            print('\nExecutando:', ' | '.join(chave_base))
            pipe = pipeline_experimento(*chave_base)

            inicio = time.perf_counter()
            busca_random = RandomizedSearchCV(
                pipe, espaco_random(nome_modelo), n_iter=N_TENTATIVAS,
                scoring='f1', cv=cv, random_state=RANDOM_STATE, n_jobs=1,
                refit=True, error_score='raise'
            )
            busca_random.fit(X_treino, y_treino)
            tempo_random = time.perf_counter() - inicio
            met = metricas_teste(busca_random.best_estimator_, X_teste, y_teste)
            resultados_metricas.append({
                'imputador': nome_imp, 'balanceamento': nome_bal, 'modelo': nome_modelo,
                'otimizador': 'Random Search', **{k: v for k, v in met.items() if k != 'matriz_confusao'}
            })
            resultados_buscas.append({
                'imputador': nome_imp, 'balanceamento': nome_bal, 'modelo': nome_modelo,
                'otimizador': 'Random Search', 'melhor_f1_cv': busca_random.best_score_,
                'tempo_segundos': tempo_random,
                'melhores_hiperparametros': json.dumps(limpar_params(busca_random.best_params_), ensure_ascii=False)
            })
            modelos_ajustados[chave_base + ('Random Search',)] = busca_random.best_estimator_

            inicio = time.perf_counter()
            def objetivo(trial):
                candidato = clone(pipe).set_params(**parametros_optuna(trial, nome_modelo))
                return cross_val_score(candidato, X_treino, y_treino, scoring='f1', cv=cv, n_jobs=1).mean()

            estudo = optuna.create_study(
                direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
            )
            estudo.optimize(objetivo, n_trials=N_TENTATIVAS, show_progress_bar=False)
            params_optuna = {f'model__{k}': v for k, v in estudo.best_params.items()}
            melhor_optuna = clone(pipe).set_params(**params_optuna).fit(X_treino, y_treino)
            tempo_optuna = time.perf_counter() - inicio
            met = metricas_teste(melhor_optuna, X_teste, y_teste)
            resultados_metricas.append({
                'imputador': nome_imp, 'balanceamento': nome_bal, 'modelo': nome_modelo,
                'otimizador': 'Optuna (TPE)', **{k: v for k, v in met.items() if k != 'matriz_confusao'}
            })
            resultados_buscas.append({
                'imputador': nome_imp, 'balanceamento': nome_bal, 'modelo': nome_modelo,
                'otimizador': 'Optuna (TPE)', 'melhor_f1_cv': estudo.best_value,
                'tempo_segundos': tempo_optuna,
                'melhores_hiperparametros': json.dumps(estudo.best_params, ensure_ascii=False)
            })
            modelos_ajustados[chave_base + ('Optuna (TPE)',)] = melhor_optuna

tabela_metricas = pd.DataFrame(resultados_metricas).sort_values('f1', ascending=False).reset_index(drop=True)
tabela_buscas = pd.DataFrame(resultados_buscas).sort_values('melhor_f1_cv', ascending=False).reset_index(drop=True)
tabela_metricas.to_csv(RESULTADOS / 'comparacao_metricas.csv', index=False, encoding='utf-8-sig')
tabela_buscas.to_csv(RESULTADOS / 'comparacao_otimizadores.csv', index=False, encoding='utf-8-sig')
print('\nMétricas no conjunto de teste:')
mostrar(tabela_metricas.round(4))
print('\nHiperparâmetros e tempo das buscas:')
mostrar(tabela_buscas.round({'melhor_f1_cv': 4, 'tempo_segundos': 2}))

# Uma linha por modelo e otimizador, como quadro-resumo para o relatório.
resumo_buscas_visual = (
    tabela_buscas.sort_values('melhor_f1_cv', ascending=False)
    .groupby(['modelo', 'otimizador'], as_index=False).first()
)
quadro = resumo_buscas_visual[[
    'modelo', 'otimizador', 'melhor_f1_cv', 'tempo_segundos', 'melhores_hiperparametros'
]].copy()
quadro['melhor_f1_cv'] = quadro['melhor_f1_cv'].map(lambda v: f'{v:.4f}')
quadro['tempo_segundos'] = quadro['tempo_segundos'].map(lambda v: f'{v:.2f} s')
quadro['melhores_hiperparametros'] = quadro['melhores_hiperparametros'].str.replace(', ', ',\n', regex=False)
fig, ax = plt.subplots(figsize=(18, 6))
ax.axis('off')
tabela_fig = ax.table(
    cellText=quadro.values, colLabels=['Modelo', 'Otimizador', 'F1-CV', 'Tempo', 'Melhores hiperparâmetros'],
    cellLoc='left', colLoc='center', loc='center', colWidths=[0.14, 0.12, 0.08, 0.08, 0.58]
)
tabela_fig.auto_set_font_size(False)
tabela_fig.set_fontsize(8)
tabela_fig.scale(1, 2.8)
for (linha, coluna), celula in tabela_fig.get_celld().items():
    if linha == 0:
        celula.set_facecolor('#3b7ca6')
        celula.set_text_props(color='white', weight='bold')
ax.set_title('Melhores hiperparâmetros por modelo e otimizador', fontsize=14, pad=18)
plt.tight_layout()
plt.savefig(RESULTADOS / '04_otimizadores_hiperparametros.png', dpi=180, bbox_inches='tight')
plt.show()

## 6. Comparações consolidadas

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=tabela_metricas, x='f1', y='modelo', hue='balanceamento', errorbar=None, ax=ax)
ax.set_title('F1 da classe sobrevivente por modelo e balanceamento')
ax.set_xlim(0, 1)
plt.tight_layout()
plt.savefig(RESULTADOS / 'comparacao_f1.png', dpi=160, bbox_inches='tight')
plt.show()

fig, eixos = plt.subplots(2, 2, figsize=(14, 9), sharey=True)
for ax, metrica in zip(eixos.flat, ['precisao', 'acuracia', 'recall', 'f1']):
    sns.barplot(
        data=tabela_metricas, x='modelo', y=metrica, hue='imputador',
        errorbar=None, palette='Set2', ax=ax
    )
    ax.set_title(f'Comparação de {metrica}')
    ax.set_xlabel('Modelo')
    ax.set_ylabel(metrica.capitalize())
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=8)
plt.tight_layout()
plt.savefig(RESULTADOS / '05_comparacao_metricas.png', dpi=170, bbox_inches='tight')
plt.show()

resumo_modelos = tabela_metricas.groupby('modelo')[['acuracia', 'precisao', 'recall', 'f1']].mean().round(4)
resumo_otimizadores = tabela_buscas.groupby('otimizador')[['melhor_f1_cv', 'tempo_segundos']].mean().round(4)
resumo_imputadores = tabela_metricas.groupby('imputador')[['acuracia', 'precisao', 'recall', 'f1']].mean().round(4)
resumo_balanceamento = tabela_metricas.groupby('balanceamento')[['precisao', 'recall', 'f1']].mean().round(4)
print('Média por modelo:'); mostrar(resumo_modelos)
print('\nMédia por otimizador:'); mostrar(resumo_otimizadores)
print('\nMédia por imputador:'); mostrar(resumo_imputadores)
print('\nMédia por balanceamento:'); mostrar(resumo_balanceamento)

## 7. Regras da melhor Árvore de Decisão

A melhor árvore é escolhida pelo F1 da validação cruzada. O teste é usado apenas para a estimativa final de generalização. Para facilitar a leitura, a árvore de apresentação é reexpressa nas unidades originais dos atributos.

In [ ]:
melhor_busca_arvore = tabela_buscas.query("modelo == 'Árvore de Decisão'").iloc[0]
melhor_linha_arvore = tabela_metricas[
    (tabela_metricas['imputador'] == melhor_busca_arvore['imputador']) &
    (tabela_metricas['balanceamento'] == melhor_busca_arvore['balanceamento']) &
    (tabela_metricas['modelo'] == melhor_busca_arvore['modelo']) &
    (tabela_metricas['otimizador'] == melhor_busca_arvore['otimizador'])
].iloc[0]
chave_arvore = (
    melhor_linha_arvore['imputador'], melhor_linha_arvore['balanceamento'],
    melhor_linha_arvore['modelo'], melhor_linha_arvore['otimizador']
)
melhor_pipeline_arvore = modelos_ajustados[chave_arvore]
# Reexpressa a árvore nas unidades originais para regras legíveis. Os exemplos
# balanceados são os mesmos; apenas desfazemos a escala antes de reajustar a cópia.
X_imp_regra = melhor_pipeline_arvore.named_steps['imputer'].transform(X_treino)
X_pad_regra = melhor_pipeline_arvore.named_steps['scaler'].transform(X_imp_regra)
X_res_regra, y_res_regra = clone(melhor_pipeline_arvore.named_steps['sampler']).fit_resample(X_pad_regra, y_treino)
X_original_regra = melhor_pipeline_arvore.named_steps['scaler'].inverse_transform(X_res_regra)
melhor_arvore = clone(melhor_pipeline_arvore.named_steps['model']).fit(X_original_regra, y_res_regra)
regras = export_text(melhor_arvore, feature_names=list(X.columns), decimals=2)
print('Configuração:', chave_arvore)
print(regras)
(RESULTADOS / 'regras_melhor_arvore.txt').write_text(
    'Configuração: ' + str(chave_arvore) + '\n\n' + regras, encoding='utf-8'
)

plt.figure(figsize=(24, 12))
plot_tree(
    melhor_arvore, feature_names=X.columns, class_names=['Morreu', 'Sobreviveu'],
    filled=True, rounded=True, fontsize=7
)
plt.title('Melhor Árvore de Decisão — Titanic')
plt.tight_layout()
plt.savefig(RESULTADOS / 'melhor_arvore.png', dpi=180, bbox_inches='tight')
plt.savefig(RESULTADOS / '06_melhor_arvore.png', dpi=180, bbox_inches='tight')
plt.show()

importancias = pd.Series(melhor_arvore.feature_importances_, index=X.columns).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
importancias.plot.barh(ax=ax, color='#3b7ca6')
ax.set_title('Importância dos atributos — Árvore de Decisão')
ax.set_xlabel('Importância')
ax.set_ylabel('Atributo')
plt.tight_layout()
plt.savefig(RESULTADOS / '07_importancia_atributos_arvore.png', dpi=170, bbox_inches='tight')
plt.show()

## 8. Matriz de confusão e relatório do melhor resultado geral

In [ ]:
melhor_busca = tabela_buscas.iloc[0]
melhor_linha = tabela_metricas[
    (tabela_metricas['imputador'] == melhor_busca['imputador']) &
    (tabela_metricas['balanceamento'] == melhor_busca['balanceamento']) &
    (tabela_metricas['modelo'] == melhor_busca['modelo']) &
    (tabela_metricas['otimizador'] == melhor_busca['otimizador'])
].iloc[0]
chave_melhor = tuple(melhor_linha[c] for c in ['imputador', 'balanceamento', 'modelo', 'otimizador'])
melhor_modelo = modelos_ajustados[chave_melhor]
pred = melhor_modelo.predict(X_teste)
matriz = confusion_matrix(y_teste, pred, labels=[0, 1])
matriz_df = pd.DataFrame(matriz, index=['Real: Morreu', 'Real: Sobreviveu'], columns=['Previsto: Morreu', 'Previsto: Sobreviveu'])
print('Melhor configuração:', chave_melhor)
mostrar(matriz_df)
print(classification_report(y_teste, pred, target_names=['Morreu', 'Sobreviveu'], zero_division=0))
sns.heatmap(matriz_df, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusão — melhor configuração')
plt.tight_layout()
plt.savefig(RESULTADOS / 'matriz_confusao_melhor_modelo.png', dpi=160, bbox_inches='tight')
plt.show()

# Matrizes dos melhores representantes de cada modelo, escolhidos pelo F1-CV.
fig, eixos = plt.subplots(1, 2, figsize=(13, 5))
for ax, nome_modelo in zip(eixos, ['Árvore de Decisão', 'Random Forest']):
    linha_busca = tabela_buscas.query('modelo == @nome_modelo').iloc[0]
    chave = tuple(linha_busca[c] for c in ['imputador', 'balanceamento', 'modelo', 'otimizador'])
    candidato = modelos_ajustados[chave]
    pred_candidato = candidato.predict(X_teste)
    matriz_candidato = confusion_matrix(y_teste, pred_candidato, labels=[0, 1])
    sns.heatmap(
        matriz_candidato, annot=True, fmt='d', cmap='viridis', cbar=True, ax=ax,
        xticklabels=['Não sobreviveu', 'Sobreviveu'],
        yticklabels=['Não sobreviveu', 'Sobreviveu']
    )
    ax.set_title(f'Matriz de confusão — {nome_modelo}')
    ax.set_xlabel('Classe prevista')
    ax.set_ylabel('Classe real')
plt.tight_layout()
plt.savefig(RESULTADOS / '08_matrizes_confusao_modelos.png', dpi=170, bbox_inches='tight')
plt.show()

## 9. Conclusão calculada a partir dos resultados

In [ ]:
melhor_modelo_nome = resumo_modelos['f1'].idxmax()
melhor_otimizador = tabela_buscas.groupby('otimizador')['melhor_f1_cv'].mean().idxmax()
otimizador_rapido = tabela_buscas.groupby('otimizador')['tempo_segundos'].mean().idxmin()
melhor_imp = resumo_imputadores['f1'].idxmax()
melhor_bal = resumo_balanceamento['f1'].idxmax()

conclusao = f'''A base possui {len(base_original)} passageiros e a classe sobrevivente representa {y.mean():.1%} dos casos, portanto foi tratada como minoritária. Entre os imputadores, {imputador_adotado} preservou melhor as distribuições pelo menor KS médio; considerando o desempenho preditivo médio, {melhor_imp} obteve o maior F1. O balanceamento com {melhor_bal} apresentou o melhor compromisso médio entre precisão e recall da classe sobrevivente. O modelo com maior F1 médio foi {melhor_modelo_nome}; a melhor configuração individual foi {chave_melhor}, com F1={melhor_linha['f1']:.3f}, precisão={melhor_linha['precisao']:.3f}, recall={melhor_linha['recall']:.3f} e acurácia={melhor_linha['acuracia']:.3f} no teste. Em validação cruzada, {melhor_otimizador} obteve a maior qualidade média, enquanto {otimizador_rapido} teve o menor tempo médio. As regras da árvore única permanecem diretamente legíveis como condições sequenciais sobre sexo, classe, idade e tarifa; no Random Forest não há uma única regra global, pois a decisão resulta do voto de muitas árvores treinadas em amostras e subconjuntos de atributos diferentes. Essa diversidade reduz variância e costuma melhorar a generalização, mas torna a explicação global menos transparente.'''
print(conclusao)
(RESULTADOS / 'conclusao.txt').write_text(conclusao, encoding='utf-8')